# 08 - Weighted Hybrid (α·SVD + (1−α)·CB)

Loads the trained SVD and Content-Based models, tunes the blend weight α on validation,
then evaluates. **Run notebooks 04 and 07 first.**

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Load the base models

In [3]:
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel
from hybrid_recsys.models.hybrid import WeightedHybrid

cb       = ContentBasedRecommender.load()
svd      = SVDModel.load()
weighted = WeightedHybrid().set_models(svd, cb)
print("loaded SVD + Content base models")

loaded SVD + Content base models


## Tune α on the validation set

In [4]:
from hybrid_recsys.evaluation.metrics import rmse

val_s = val.sample(min(50_000, len(val)), random_state=RANDOM_STATE)   # fast representative sweep
alphas = np.round(np.arange(0.0, 1.05, 0.05), 2)
rmses = []
for a in alphas:
    weighted.alpha = float(a)
    p = np.array([weighted.predict(r.userId, r.movieId, user_ratings_map.get(r.userId, {}))
                  for r in val_s.itertuples()])
    rmses.append(rmse(val_s["rating"].to_numpy(), p))

best = float(alphas[int(np.argmin(rmses))])
weighted.alpha = best
weighted.save()
print(f"best α (SVD weight) = {best}")

best α (SVD weight) = 0.95


## α-sweep curve

In [5]:
fig = px.line(x=alphas, y=rmses, markers=True,
              title="Validation RMSE vs α  (α=1 -> pure SVD, α=0 -> pure CB)",
              labels={"x": "α (SVD weight)", "y": "validation RMSE"})
fig.add_vline(x=best, line_dash="dash", line_color="red")
save_fig(fig, "eval_weighted_alpha")

## Evaluate — compute & record metrics

In [6]:
w_predict = lambda u, i: weighted.predict(u, i, user_ratings_map.get(u, {}))
m, preds = full_metrics(w_predict, test, test_sample, train_val, all_movie_ids,
                        n_negatives=N_NEGATIVES, random_state=RANDOM_STATE)
save_metric("Weighted Hybrid", m)
print(f"RMSE={m['rmse']}  MAE={m['mae']}  F1@10={m['k10']['f1']}")

RMSE=0.8095  MAE=0.6074  F1@10=0.3541


## Ranking metrics @ K

In [7]:
save_fig(ranking_curve(m, "Weighted Hybrid"), "eval_weighted_ranking")

## Example recommendations for the demo user

In [10]:
hist, recs = show_example(w_predict)
display(recs[["clean_title", "genres", "pred"]])

,clean_title,genres,pred
0,"Haunting, The",Horror|Thriller,4.530
1,"Story of the Late Chrysanthemums, The (Zangiku...",Drama,4.472
2,Satan's Tango (Sátántangó),Drama,4.420
3,Land of Silence and Darkness (Land des Schweig...,Documentary,4.395
4,Trust,Comedy|Drama|Romance,4.378
5,Patriotism (Yûkoku),Drama,4.370
6,Phantom Thread,Drama|Romance,4.367
7,"Bonheur, Le",Drama,4.362
8,"Trap: What Happened to Our Dream of Freedom, The",Documentary,4.360
9,The Boy and the World,Animation,4.326


## Takeaway

α converges high (≈0.9) - CB is the weaker signal, so the blend is mostly SVD. Safe and never worse than SVD, but a single global weight can't adapt per item (→ stacking).